# Limpieza · demanda real por empresa de SIMEM (`14fabb` desagregado)

El detalle de SIMEM: **7,4 millones de filas**, una por hora × empresa (77
agentes) × tipo de mercado × **versión de liquidación**. Son 113 combinaciones
`(CodigoSICAgente, TipoMercado)`.

Lo que se encontró al mirarlo:

- varias versiones de liquidación por hora (hasta 5 en enero de 2026): cada
  empresa aparece repetida una vez por versión;
- **101 567 ceros** una vez colapsadas las versiones (167 102 contando cada
  versión por separado), en 36 grupos; alguna empresa está en cero más del 80 %
  de las horas. En el total nacional un cero sería imposible, pero en una
  empresa concreta una hora sin consumo es plausible, así que se conservan;
- ni negativos ni nulos.

La limpieza tiene dos pasos:

1. **Colapsar versiones**: una sola por hora, la más definitiva, con
   `ingesta.normalizar.simem_colapsar_versiones()` —la misma que usa el total
   nacional—. Se comprueba año a año que no se pierde ninguna empresa.
2. **Limpiar por grupos**: cada `(empresa, mercado)` como una serie propia, con
   `limpieza.grupos.limpiar_por_grupos()`, igual que CIIU.

In [1]:
import sys, warnings
from pathlib import Path

# Sube desde la carpeta del notebook hasta la raiz del proyecto, la que contiene
# src/ingesta: funciona igual en notebooks/ que en notebooks/limpieza/<dataset>/.
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "ingesta").is_dir())
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)

DATASETS = RAIZ / "datasets"
print("raiz     :", RAIZ)
print("datasets :", DATASETS, "->", "existe" if DATASETS.exists() else "FALTA")

import json, time
import pyarrow as pa
import pyarrow.parquet as pq

raiz     : C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER
datasets : C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER\datasets -> existe


## Carga y colapso de versiones, año a año

Se procesa un año cada vez: colapsar sobre el año o sobre todo el histórico da
lo mismo, porque la versión se elige hora a hora. Para cada año se comprueba
que el número de combinaciones empresa-mercado de cada hora es el mismo antes y
después de colapsar.

In [2]:
from pandas.api.types import union_categoricals
from ingesta.normalizar import simem_colapsar_versiones

GRUPO = ["CodigoSICAgente", "TipoMercado"]
t0 = time.time()
partes, control = [], []
for ruta in sorted(DATASETS.glob("simem/simem_14fabb_20*.csv")):
    d = pd.read_csv(
        ruta, usecols=["timestamp", *GRUPO, "Version", "Valor"],
        dtype={"CodigoSICAgente": "category", "TipoMercado": "category"},
    )
    d["timestamp"] = pd.to_datetime(d["timestamp"], format="ISO8601")
    d["FechaHora"] = d["timestamp"]   # la columna que usa el colapso

    antes = d.drop_duplicates(["FechaHora", *GRUPO]).groupby("FechaHora").size()
    colapsado = simem_colapsar_versiones(d, GRUPO)
    despues = colapsado.groupby("FechaHora").size()
    perdidas = int((antes - despues.reindex(antes.index, fill_value=0)).sum())
    if perdidas:
        raise RuntimeError(f"{ruta.name}: el colapso dejó fuera {perdidas} combinaciones empresa-hora")

    control.append({
        "archivo": ruta.name, "filas_con_versiones": len(d), "filas_colapsadas": len(colapsado),
        "versiones": sorted(d.Version.unique()), "combinaciones_perdidas": perdidas,
    })
    partes.append(colapsado.drop(columns="FechaHora"))
    del d, colapsado

simem = pd.DataFrame({
    "timestamp": pd.concat([p.timestamp for p in partes], ignore_index=True),
    "CodigoSICAgente": union_categoricals([p.CodigoSICAgente for p in partes]),
    "TipoMercado": union_categoricals([p.TipoMercado for p in partes]),
    "Version": pd.concat([p.Version for p in partes], ignore_index=True),
    "Valor": pd.concat([p.Valor for p in partes], ignore_index=True),
})
del partes
print(f"{len(simem):,} filas tras colapsar, en {time.time() - t0:.0f} s")
pd.DataFrame(control)

4,432,560 filas tras colapsar, en 12 s


,archivo,filas_con_versiones,filas_colapsadas,versiones,combinaciones_perdidas
0,simem_14fabb_2021.csv,755640,755640,"[TX4, TX5]",0
1,simem_14fabb_2022.csv,769416,769416,"[TX3, TX4, TX5, TX6]",0
2,simem_14fabb_2023.csv,784680,784680,"[TX4, TX5, TX6]",0
3,simem_14fabb_2024.csv,772632,772632,"[TX4, TX5, TX6, TX7, TX8]",0
4,simem_14fabb_2025.csv,2226960,798384,"[TX2, TX3, TX4, TX5, TXF, TXR]",0
5,simem_14fabb_2026.csv,2093856,551808,"[TX2, TX3, TX4, TXF, TXR]",0


## Diagnóstico

In [3]:
g = simem.groupby(GRUPO, observed=True)
filas = g.size()
vida = g.timestamp.agg(["min", "max"])
esperadas = ((vida["max"] - vida["min"]) / pd.Timedelta(hours=1) + 1).astype(int)
ceros = g.Valor.apply(lambda s: int((s == 0).sum()))

print(f"empresas                       : {simem.CodigoSICAgente.nunique()}")
print(f"grupos (empresa, mercado)      : {len(filas)}")
print(f"filas por grupo                : min {filas.min():,} · mediana {int(filas.median()):,} · máx {filas.max():,}")
print(f"grupos con horas sin fila      : {int((esperadas > filas).sum())} ({int((esperadas - filas).sum()):,} horas)")
print(f"ceros                          : {int((simem.Valor == 0).sum()):,} en {int((ceros > 0).sum())} grupos")
print(f"negativos / nulos              : {int((simem.Valor < 0).sum())} / {int(simem.Valor.isna().sum())}")
por_hora = simem.groupby("timestamp").size()
print(f"combinaciones por hora         : {por_hora.min()} .. {por_hora.max()}")

empresas                       : 77
grupos (empresa, mercado)      : 113
filas por grupo                : min 120 · mediana 49,776 · máx 49,776
grupos con horas sin fila      : 1 (9,600 horas)
ceros                          : 101,567 en 36 grupos
negativos / nulos              : 0 / 0
combinaciones por hora         : 84 .. 94


In [4]:
print("Grupos con más ceros (en proporción):")
(ceros / filas).rename("pct_ceros").mul(100).round(1).sort_values(ascending=False).head(8).to_frame()

Grupos con más ceros (en proporción):


,,pct_ceros
CodigoSICAgente,TipoMercado,
BCCC,No Regulado,81.9
GSAC,No Regulado,75.3
TRPC,No Regulado,54.0
CDSC,No Regulado,50.0
RPEC,No Regulado,12.2
ESVC,Regulado,8.0
ESOC,Regulado,6.8
SCEC,No Regulado,6.4


## Limpieza por grupos

Cada `(empresa, mercado)` se limpia como una serie propia y se escribe en
cuanto está lista. La columna `version` no es constante dentro de un grupo —cambia
con los años—, así que en las horas creadas para completar la rejilla queda
vacía: no se sabe qué versión habría tenido una hora que no llegó.

In [5]:
from limpieza.grupos import limpiar_por_grupos, fila_de_resumen, resumir

SALIDA = DATASETS / "limpios"
SALIDA.mkdir(parents=True, exist_ok=True)
destino = SALIDA / "simem_desagregado.parquet"
temporal = destino.with_name(destino.name + ".tmp")

ESQUEMA = pa.schema([
    ("timestamp", pa.timestamp("ns", tz="America/Bogota")),
    ("codigo_sic_agente", pa.string()), ("tipo_mercado", pa.string()),
    ("version", pa.string()), ("valor", pa.float64()), ("origen_valor", pa.string()),
    ("imputado", pa.bool_()), ("hueco_horas", pa.int64()),
    ("z_estacional", pa.float64()), ("atipico", pa.bool_()),
    ("atipico_evaluable", pa.bool_()), ("atipico_iqr", pa.bool_()),
    ("atipico_global", pa.bool_()), ("atipico_iqr_global", pa.bool_()),
    ("periodo_atipico", pa.bool_()),
])

filas_resumen = []
t0 = time.time()
with pq.ParquetWriter(temporal, ESQUEMA, compression="zstd") as escritor:
    for n, (etiqueta, limpio, registro) in enumerate(limpiar_por_grupos(simem, GRUPO), 1):
        assert not limpio.timestamp.duplicated().any(), f"horas repetidas en {etiqueta}"
        tabla = limpio[ESQUEMA.names].astype({"hueco_horas": "int64"})
        escritor.write_table(pa.Table.from_pandas(tabla, schema=ESQUEMA, preserve_index=False))
        filas_resumen.append(fila_de_resumen(etiqueta, limpio, registro))
        if n % 30 == 0:
            print(f"  {n:>3} grupos · {time.time() - t0:.0f} s")
temporal.replace(destino)

total = resumir(filas_resumen)
print(f"{total['n_grupos']} grupos limpiados en {time.time() - t0:.0f} s")
total

   30 grupos · 19 s


   60 grupos · 38 s


   90 grupos · 55 s


113 grupos limpiados en 65 s


{'n_grupos': 113,
 'filas_iniciales': 4432560,
 'filas_finales': 4442160,
 'horas_creadas': 9600,
 'observados': 4432560,
 'interpolados': 0,
 'faltantes': 9600,
 'atipicos': 309704,
 'no_evaluables': 389551,
 'grupos_con_interpolados': 0,
 'grupos_con_faltantes': 1,
 'pct_observado': 99.7839}

## Resultado

In [6]:
grupos = pd.DataFrame(filas_resumen)
print(f"observados  : {total['observados']:>11,}  ({total['pct_observado']} %)")
print(f"interpolados: {total['interpolados']:>11,}  en {total['grupos_con_interpolados']} grupos (huecos de hasta 3 h)")
print(f"faltantes   : {total['faltantes']:>11,}  en {total['grupos_con_faltantes']} grupos (huecos largos, se quedan NaN)")
print(f"atípicos    : {total['atipicos']:>11,}  (causales, dentro de cada grupo)")
print(f"horas creadas para completar la rejilla de cada grupo: {total['horas_creadas']:,}")
print()
print("Grupos con más horas sin valor:")
grupos.sort_values("faltantes", ascending=False).head(8)[
    ["CodigoSICAgente", "TipoMercado", "desde", "hasta", "faltantes", "tramos_no_imputados", "interpolados"]
]

observados  :   4,432,560  (99.7839 %)
interpolados:           0  en 0 grupos (huecos de hasta 3 h)
faltantes   :       9,600  en 1 grupos (huecos largos, se quedan NaN)
atípicos    :     309,704  (causales, dentro de cada grupo)
horas creadas para completar la rejilla de cada grupo: 9,600

Grupos con más horas sin valor:


,CodigoSICAgente,TipoMercado,desde,hasta,faltantes,tramos_no_imputados,interpolados
43,ENDC,Regulado,2021-01-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,9600,2,0
84,TPLC,No Regulado,2021-01-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,0,0,0
82,RTQC,Regulado,2021-01-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,0,0,0
81,RTQC,No Regulado,2021-01-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,0,0,0
80,RTAC,Regulado,2021-01-01 00:00:00-05:00,2022-06-08 23:00:00-05:00,0,0,0
79,RTAC,No Regulado,2021-01-01 00:00:00-05:00,2024-04-30 23:00:00-05:00,0,0,0
78,RQEC,No Regulado,2021-01-01 00:00:00-05:00,2023-05-13 23:00:00-05:00,0,0,0
77,RPEC,No Regulado,2021-01-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,0,0,0


## Comprobaciones

La más fuerte: sumando por hora los valores **observados** del detalle limpio
tiene que salir exactamente la serie nacional de SIMEM, que se construyó desde
el mismo detalle. Si la limpieza hubiera perdido, duplicado o movido una sola
fila, no cuadraría.

In [7]:
meta = pq.ParquetFile(destino).metadata
assert meta.num_rows == total["filas_finales"], "el Parquet no tiene las filas esperadas"
assert total["filas_iniciales"] == len(simem), "no se procesaron todas las filas"
assert total["observados"] + total["interpolados"] + total["faltantes"] == total["filas_finales"]
assert total["filas_finales"] - total["filas_iniciales"] == total["horas_creadas"]

detalle = pq.read_table(destino, columns=["timestamp", "valor", "origen_valor"],
                        read_dictionary=["origen_valor"]).to_pandas()
reconstruida = detalle[detalle.origen_valor == "observado"].groupby("timestamp").valor.sum()
del detalle

nacional = pd.read_csv(DATASETS / "simem" / "simem_14fabb_nacional.csv", parse_dates=["fecha_hora"])
nacional = nacional.set_index(nacional.fecha_hora.dt.tz_convert("America/Bogota")).valor_kwh
comparacion = pd.concat([reconstruida.rename("desde_detalle"), nacional.rename("nacional")], axis=1)
diferencia = (comparacion.desde_detalle - comparacion.nacional).abs()

assert comparacion.notna().all().all(), "hay horas en una serie y no en la otra"
assert diferencia.max() < 1.0, f"el detalle no reconstruye la serie nacional: {diferencia.max():,.3f} kWh"
print(f"OK: {len(simem):,} filas de entrada -> {meta.num_rows:,} en el Parquet")
print(f"OK: la suma del detalle reproduce las {len(comparacion):,} horas de la serie nacional")
print(f"    diferencia máxima: {diferencia.max():.6f} kWh")

OK: 4,432,560 filas de entrada -> 4,442,160 en el Parquet
OK: la suma del detalle reproduce las 49,776 horas de la serie nacional
    diferencia máxima: 0.000000 kWh


## Registro

In [8]:
registro_simem = {
    "version_formato": 1,
    "conjunto": "simem_14fabb_desagregado",
    "momento": pd.Timestamp.now().isoformat(timespec="seconds"),
    "archivo": destino.name,
    "politica": ("una version de liquidacion por hora (la mas definitiva), comprobando que no se "
                 "pierde ninguna combinacion; limpieza por (CodigoSICAgente, TipoMercado); rejilla "
                 "de cada grupo entre su primera y su ultima hora; interpolacion solo en huecos de "
                 "hasta 3 h; atipicos causales dentro de cada grupo; los ceros se conservan"),
    "colapso_de_versiones": control,
    "resumen": total,
    "grupos": filas_resumen,
}
with open(SALIDA / "registro_simem_desagregado.json", "w", encoding="utf-8") as f:
    json.dump(registro_simem, f, ensure_ascii=False, indent=2, default=str)
print("registro ->", SALIDA / "registro_simem_desagregado.json")

registro -> C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER\datasets\limpios\registro_simem_desagregado.json
